# Humanization: protocollo v2 su 126 fonti

Nuove condizioni, controlli nulli, Binoculars e NLI contestuale. Iniziare con MODE="smoke"; dopo il collaudo passare a "study" mantenendo il preset. I vecchi pilot restano conservati.


## 1. Configurazione e GPU
T4 ×2 e Internet attivo. Le condizioni sono nominate esplicitamente: baseline 3B, 14B NF4 e due protocolli DIPPER XXL.


In [ ]:
from pathlib import Path
import gc, hashlib, json, os, shutil, subprocess, sys

MODE = "smoke"                     # "smoke", "pilot", "study"
RUN_SUFFIX = "_protocol_v2"                    # ad es. "_dipper" per una nuova condizione
ACTIVE_SYSTEMS = ["identity", "qwen_humanize", "qwen14b_humanize", "dipper", "dipper_context3"]
JUDGE_MODEL = "qwen/qwen3.8-27b"  # exact requested ID; authenticated catalog check required
RUN_JUDGE_VALIDATION = True
RUN_STUDY_JUDGE = True
RUN_BINOCULARS = True
REPO_REF = "main"                  # si registra il commit effettivamente risolto
assert MODE in {"smoke", "study"}
assert Path("/kaggle/working").exists(), "Questo notebook è preparato per Kaggle."
TMP = Path("/kaggle/temp/humanization_runtime")
TMP.mkdir(parents=True, exist_ok=True)
REPO = Path("/kaggle/temp") / ("Humanization_v2_" + MODE + RUN_SUFFIX)
RUN = Path("/kaggle/working") / ("humanization_v2_" + MODE + RUN_SUFFIX)
RUN.mkdir(parents=True, exist_ok=True)
for key, value in {
    "HF_HOME": str(TMP / "huggingface"),
    "HF_HUB_CACHE": str(TMP / "huggingface" / "hub"),
    "HF_XET_CACHE": str(TMP / "huggingface" / "xet"),
    "HF_HUB_DISABLE_XET": "1",
    "TMPDIR": str(RUN / "_tmp"),
    "TMP": str(RUN / "_tmp"),
    "TEMP": str(RUN / "_tmp"),
    "NLTK_DATA": str(TMP / "nltk_data"),
    "MPLCONFIGDIR": str(TMP / "matplotlib"),
    "XDG_CACHE_HOME": str(TMP / "cache"),
    "TOKENIZERS_PARALLELISM": "false", "OMP_NUM_THREADS": "2",
    "USE_TF": "0", "USE_FLAX": "0", "USE_TORCH": "1",
    "HF_HUB_DOWNLOAD_TIMEOUT": "120",
}.items():
    os.environ[key] = value

# Redirect legacy overrides if the Kaggle image inherited them.
for variable in ("TRANSFORMERS_CACHE", "HUGGINGFACE_HUB_CACHE"):
    if variable in os.environ:
        os.environ[variable] = str(TMP / "huggingface" / "hub")

# Keep model/cache/temp writes in the writable Kaggle runtime.
for directory in [TMP / "huggingface" / "hub", TMP / "huggingface" / "xet", RUN / "_tmp"]:
    directory.mkdir(parents=True, exist_ok=True)
    probe = directory / ".humanization_write_probe"
    probe.write_text("ok")
    probe.unlink()

import torch
from packaging.version import Version
assert torch.cuda.is_available(), "Abilita una GPU nelle impostazioni Kaggle."
assert Version(torch.__version__.split("+")[0]) >= Version("2.6"), "Serve l'immagine Kaggle con PyTorch CUDA >= 2.6 per RADAR."
print("PyTorch:", torch.__version__)
print("GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("Disco temporaneo libero (GiB):", round(shutil.disk_usage(TMP).free / 2**30, 1))

if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/FrancescoM00/Humanization.git", str(REPO)], check=True)
subprocess.run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
previous = RUN / "runtime.json"
if previous.exists():
    assert json.loads(previous.read_text())["repository_commit"] == COMMIT, "Il codice è cambiato: scegli un RUN_SUFFIX nuovo."
print("Repository commit:", COMMIT)

assert torch.cuda.device_count() >= 2, "Binoculars requires two distinct GPUs: select T4 x2."


## 2. Dipendenze e config separato
Riusa CUDA e cache della sessione; non installare il lockfile CPU.


In [ ]:
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["USE_TORCH"] = "1"

VENV = TMP / "venv"
PYTHON = VENV / "bin/python"
# Kaggle may not provide ensurepip. Reapply creation even after a partial
# attempt: bin/python can exist while system-site-packages is still disabled.
subprocess.run([sys.executable, "-m", "venv", "--without-pip",
                "--system-site-packages", str(VENV)], check=True)
pip_probe = subprocess.run([str(PYTHON), "-m", "pip", "--version"],
                           capture_output=True, text=True)
if not (VENV / "bin/pip").is_file() or pip_probe.returncode != 0:
    # Base pip manages the target venv; it does not replace Kaggle CUDA torch.
    subprocess.run([sys.executable, "-m", "pip", "--python", str(PYTHON),
                    "install", "--upgrade", "--ignore-installed", "pip"], check=True)
requirements = REPO / "requirements-kaggle.txt"
stamp = VENV / "requirements.sha256"
expected = hashlib.sha256(requirements.read_bytes()).hexdigest()
if not stamp.exists() or stamp.read_text() != expected:
    subprocess.run([str(PYTHON), "-m", "pip", "install", "-r", str(requirements)], check=True)
    stamp.write_text(expected)
subprocess.run([str(PYTHON), "-c", "import nltk,os; nltk.download('cmudict',download_dir=os.environ['NLTK_DATA'],raise_on_error=True)"], check=True)

# Check concrete model imports in a fresh process, without downloading weights.
runtime_probe = 'import os\nos.environ["USE_TF"] = "0"\nos.environ["USE_FLAX"] = "0"\nos.environ["USE_TORCH"] = "1"\nimport torch, spacy, textstat\nfrom transformers import (Qwen2ForCausalLM, RobertaForSequenceClassification,\n                          DebertaV2ForSequenceClassification, T5ForConditionalGeneration)\nfrom sentence_transformers import SentenceTransformer\nfrom bert_score import BERTScorer\nnlp = spacy.load("en_core_web_sm")\nassert len(nlp("This is a short English passage.")) > 0\nprint("Import e modello spaCy: OK")\nassert torch.cuda.is_available(), "CUDA non disponibile nella venv"\nx = torch.ones((16, 16), device="cuda", dtype=torch.float16)\ny = x @ x\ntorch.cuda.synchronize()\nassert bool(torch.isfinite(y).all()), "Il controllo CUDA ha prodotto valori non finiti"\nprint("CUDA nella venv:", torch.__version__)\nprint("Kernel CUDA FP16: OK")\n'
subprocess.run([str(PYTHON), "-c", runtime_probe], check=True)

# Check the installed CUDA quantization backend before a large DIPPER download.
if any(s["name"] in ACTIVE_SYSTEMS and s.get("load_in_4bit")
       for s in json.loads((REPO / "kaggle/config-study-v2.json").read_text())["humanizers"]):
    quantization_probe = 'import torch\nimport bitsandbytes as bnb\nassert torch.cuda.is_available(), "Abilitare una GPU Kaggle"\nlayer = bnb.nn.Linear4bit(64, 64, bias=False,\n                         compute_dtype=torch.float16,\n                         compress_statistics=True,\n                         quant_type="nf4").to("cuda:0").eval()\nx = torch.ones((1, 64), device="cuda:0", dtype=torch.float16)\nwith torch.inference_mode():\n    y = layer(x)\ntorch.cuda.synchronize()\nassert bool(torch.isfinite(y).all()), "Il controllo NF4 ha prodotto valori non finiti"\nprint("Backend CUDA 4-bit: OK")\n'
    subprocess.run([str(PYTHON), "-c", quantization_probe], check=True)

def run(script, *arguments):
    scratch = RUN / "_tmp"
    scratch.mkdir(parents=True, exist_ok=True)
    child_env = dict(os.environ)
    child_env.update({name: str(scratch) for name in ("TMPDIR", "TMP", "TEMP")})
    subprocess.run([str(PYTHON), str(REPO / "scripts" / script), *map(str, arguments)],
                   cwd=REPO, env=child_env, check=True)

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]

CONFIG = RUN / "config.json"
config_source = REPO / "kaggle/config-study-v2.json"
if CONFIG.exists():
    assert CONFIG.read_bytes() == config_source.read_bytes(), "Config cambiata: scegli un RUN_SUFFIX nuovo."
else:
    shutil.copy2(config_source, CONFIG)
config = json.loads(CONFIG.read_text())
runtime = {"repository_commit": COMMIT, "mode": MODE, "active_systems": ACTIVE_SYSTEMS,
           "torch": torch.__version__, "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
           "requirements_sha256": expected,
           "config_sha256": hashlib.sha256(CONFIG.read_bytes()).hexdigest()}
if previous.exists():
    assert json.loads(previous.read_text())["active_systems"] == ACTIVE_SYSTEMS, "Sistemi cambiati: scegli un RUN_SUFFIX nuovo."
previous.write_text(json.dumps(runtime, indent=2))
(RUN / "pip_freeze.txt").write_text(subprocess.check_output([str(PYTHON), "-m", "pip", "freeze"], text=True))
print("Configurazione salvata:", CONFIG)


## 3. Groq: catalogo, nessun ragionamento e prova sui giudizi esistenti
Abilitare GROQ_API_KEY nei Secrets del notebook. L’ID richiesto viene verificato e non sostituito. La prova usa sei fonti dei 24 voti dell’assistente: concordanza AI–AI, non gold umana.


In [ ]:
# Only this subprocess environment receives the key; it is never exported to files.
if RUN_JUDGE_VALIDATION or RUN_STUDY_JUDGE:
    from kaggle_secrets import UserSecretsClient
    os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")
    run("judge_naturalness.py", "--check-model", "--model", JUDGE_MODEL)

if RUN_JUDGE_VALIDATION:
    # Validate the requested judge on the six sources of the existing 24 ratings.
    # Load archived pilot outputs; no model regenerates these reference texts.
    import urllib.request, zipfile, io
    reference_dir = TMP / "judge_pilot_reference"
    reference_dir.mkdir(exist_ok=True)
    references = [
        ("exports/kaggle_pilot/humanization_pilot.zip",
         "7a35ca521ccc4e2147b42e479fe2ac6bde1ef676ddb462137a27af046960f272",
         "rewriting_cuda0/rewrites.jsonl", "qwen3b.jsonl"),
        ("exports/kaggle_dipper_pilot/humanization_pilotdipper.zip",
         "21a66ed1cb1d3744be742a79f4fb7b924668d21200ed11033364ac48a18df31d",
         "rewriting/rewrites.jsonl", "dipper_passage.jsonl"),
    ]
    reference_paths = []
    for relative, digest, member, name in references:
        target = reference_dir / name
        if not target.exists():
            url = f"https://raw.githubusercontent.com/FrancescoM00/Humanization/{COMMIT}/{relative}"
            with urllib.request.urlopen(url, timeout=120) as response:
                body = response.read()
            assert hashlib.sha256(body).hexdigest() == digest
            with zipfile.ZipFile(io.BytesIO(body)) as archive:
                target.write_bytes(archive.read(member))
        reference_paths.append(target)
    ref_args = [item for path in reference_paths for item in ("--rewrites", path)]
    validation_dir = RUN / "judge_pilot_validation"
    run("judge_naturalness.py", *ref_args, "--model", JUDGE_MODEL,
        "--humanizer-a", "qwen_humanize", "--humanizer-b", "dipper",
        "--source-key", REPO / "exports/kaggle_dipper_pilot/naturalness_paired_24_key.csv",
        "--max-pairs", 6, "--max-requests", 16, "--execute", "--output", validation_dir,
        "--ratings", REPO / "results_naturalness_assistant24/ratings.csv",
        "--rating-key", REPO / "exports/kaggle_dipper_pilot/naturalness_paired_24_key.csv",
        "--label-source", "assistant_proxy")
    validation_run = json.loads((validation_dir / "judge_run.json").read_text())
    print("Giudice:", validation_run["status"], "POST:", validation_run["generation_post_attempts"])
    print("Concordanza AI-AI, non validazione umana:",
          json.loads((validation_dir / "rating_validation.json").read_text()))
    assert validation_run["status"] == "completed", "Giudice incompleto: leggere judge_run.json prima di proseguire."


## 4. Coorti e politica sulle consegne
Tutti gli input restano nel risultato primario. La scheda di tipo usa solo gli originali; nessuna esclusione automatica basata sugli score.


In [ ]:
RAW_DIR = TMP / "raw"
required_csv = ["Essay_LLMs.csv", "Reuters_LLMs.csv", "WP_LLMs.csv"]
if not all((RAW_DIR / name).is_file() for name in required_csv):
    run("download_mgtbench.py", "--source", "mirror", "--output", RAW_DIR)
COHORTS = RUN / "cohorts"
run("prepare_humanizer_pilot.py", "--raw-dir", RAW_DIR, "--output", COHORTS)
COHORT = COHORTS / (MODE + ".jsonl")
cohort = read_jsonl(COHORT)
print("Umani di calibrazione:", sum(r["split"] == "calibration" for r in cohort))
print("Fonti di test:", sum(r["split"] == "test" for r in cohort))

run("prepare_study_v2.py", "--cohort", COHORT, "--panel", MODE, "--output", RUN / "study_protocol")
assert sum(r["split"] == "test" for r in cohort) == (3 if MODE == "smoke" else 126)


## 5. Riscritture
Cinque condizioni incluse le identità: attesi 15 esiti smoke o 630 study. Il protocollo DIPPER a tre frasi usa prefisso cumulativo e budget per finestra dichiarato.


In [ ]:
REWRITES = RUN / "rewriting"
selected_args = [item for name in ACTIVE_SYSTEMS for item in ("--only", name)]
run("run_humanizers.py", "--cohort", COHORT, "--config", CONFIG,
    "--output", REWRITES, "--device", "cuda:0", *selected_args)
rewrites = read_jsonl(REWRITES / "rewrites.jsonl")
from collections import Counter
print("Esiti:", Counter((r["humanizer"], r["status"]) for r in rewrites))
for row in [r for r in rewrites if r["humanizer"] != "identity"][:2]:
    print("\nSISTEMA:", row["humanizer"], "DOMINIO:", row["domain"], "ESITO:", row["status"])
    print("ORIGINALE:", row["original"])
    print("RISCRITTURA:", row["modified"])


## 6. Controlli nulli e classificatori
Prefissi con conteggio alfabetico esatto, rumore lessicale minimo e loro combinazione. Le frasi incomplete e i target più lunghi dell’originale sono marcati; nessun padding.


In [ ]:
NULLS = RUN / "null_controls"
run("prepare_null_controls.py", "--cohort", COHORT,
    "--rewrites", REWRITES / "rewrites.jsonl", "--output", NULLS, "--include-rewrites")
DETECTION_PAIRS = NULLS / "rewrites.jsonl"
print("Controlli e riscritture:", len(read_jsonl(DETECTION_PAIRS)))
DETECTION = RUN / "detection"
run("run_detectors.py", "--cohort", COHORT, "--rewrites", DETECTION_PAIRS,
    "--config", CONFIG, "--output", DETECTION, "--device", "cuda:0")
import pandas as pd
summary = pd.read_csv(DETECTION / "detection_summary.csv")
display(summary.loc[summary.domain == "overall", ["detector", "humanizer", "n_test_sources",
    "human_test_fpr", "original_machine_tpr", "baseline_detected", "attack_successes", "asr",
    "asr_ci95_low", "asr_ci95_high", "generations_failed"]])
display(pd.read_csv(DETECTION / "detection_macro_summary.csv"))


## 7. Binoculars su due GPU
Ratio grezzo, AI se basso, soglia sui soli 300 umani. Due Falcon 7B FP16, nessuna quantizzazione; caricamento reale da verificare nello smoke. Il voto RoBERTa rimane una sintesi separata.


In [ ]:
ZERO_SHOT = RUN / "zero_shot"
if RUN_BINOCULARS:
    # Each Falcon 7B stays on one GPU in FP16; writer processes have exited.
    run("run_zero_shot_detectors.py", "--cohort", COHORT, "--rewrites", DETECTION_PAIRS,
        "--config", CONFIG, "--output", ZERO_SHOT,
        "--observer-device", "cuda:0", "--performer-device", "cuda:1")
    zero_summary = pd.read_csv(ZERO_SHOT / "zero_shot_summary.csv")
    display(zero_summary[zero_summary.domain == "overall"])


## 8. Stile sulle riscritture effettive
Distanze descrittive, senza equipararle a naturalezza.


In [ ]:
STYLE = RUN / "style"
run("score_humanizer_style.py", "--cohort", COHORT,
    "--pairs", REWRITES / "pairs.jsonl", "--output", STYLE)
style = pd.read_csv(STYLE / "style_pairs.csv")
display(style.head())
print("Moduli di revisione:")
print(*[p.name for p in STYLE.glob("*form.csv")], sep="\n")


## 9. Metriche semantiche storiche
MiniLM, BERTScore e stesso DeBERTa-v3 base float32. I controlli nulli non sono etichette semantiche gold.


In [ ]:
SEMANTIC = RUN / "semantic"
SCORES = SEMANTIC / "scores"
# Writable shared cache outside RUN, so weights stay out of the result ZIP.
MODELS = RUN.parent / ".humanization_semantic_models"
MODELS.mkdir(parents=True, exist_ok=True)
for kind in config["semantic_models"]:
    old_model = TMP / "models" / kind
    new_model = MODELS / kind
    if old_model.is_dir() and not new_model.exists():
        shutil.copytree(old_model, new_model)
SEMANTIC.mkdir(exist_ok=True)
source_pairs = REWRITES / "pairs.jsonl"
target_pairs = SEMANTIC / "pairs.jsonl"
if target_pairs.exists():
    assert target_pairs.read_bytes() == source_pairs.read_bytes(), "Le coppie sono cambiate: usa una directory nuova."
else:
    shutil.copy2(source_pairs, target_pairs)
assert read_jsonl(target_pairs), "Nessuna coppia valida: controlla prima le generazioni."
NLI_BATCH_SIZE = 4
pairs_hash = hashlib.sha256(target_pairs.read_bytes()).hexdigest()

def completed_stage(stem):
    meta, scores = SCORES / (stem + "_run.json"), SCORES / (stem + ".csv")
    return (meta.is_file() and scores.is_file()
            and json.loads(meta.read_text()).get("pairs_sha256") == pairs_hash)

for stage in ["lexical", "embedding", "bertscore", "nli"]:
    if completed_stage(stage):
        print("Riutilizzo:", stage)
        continue
    if stage != "lexical":
        run("download_semantic_models.py", "--model", stage, "--revision",
            config["semantic_models"][stage], "--output", MODELS)
    run("run_semantic_experiment.py", "--stage", stage, "--data-dir", SEMANTIC,
        "--output", SCORES, "--models", MODELS, "--device", "cuda:0",
        "--nli-batch-size", NLI_BATCH_SIZE, "--threads", 2)
for kind, stem in [("cosine", "aligned_cosine"), ("bertscore", "aligned_bertscore")]:
    if completed_stage(stem):
        print("Riutilizzo:", stem)
        continue
    if kind == "bertscore":
        bert_dir = MODELS / "bertscore"
        if not (bert_dir / "config.json").is_file() or not any((bert_dir / name).is_file() for name in ["model.safetensors", "pytorch_model.bin"]):
            run("download_semantic_models.py", "--model", "bertscore", "--revision",
                config["semantic_models"]["bertscore"], "--output", MODELS)
    run("score_aligned_similarity.py", "--stage", kind, "--pairs", target_pairs,
        "--alignment", SEMANTIC / "alignment.json", "--output", SCORES,
        "--model", MODELS / "bertscore", "--device", "cuda:0", "--threads", 2)


## 10. NLI contestuale e confronto con i controlli nulli
Unità d’ipotesi identiche; premesse singole, finestre fino a due/tre unità, documento intero soltanto quando entra nel budget. Nessuna soglia semantica trasferita.


In [ ]:
CONTEXT = RUN / "semantic_context"
run("score_contextual_nli.py", "--pairs", target_pairs, "--alignment", SEMANTIC / "alignment.json",
    "--model", MODELS / "nli", "--output", CONTEXT, "--device", "cuda:0",
    "--batch-size", 4, "--threads", 2, "--modes", "single_all", "window2", "window3", "document")
display(pd.read_csv(CONTEXT / "contextual_nli.csv").head(12))
# Incomplete full-document coverage remains missing, not an average over easy units.

thresholds = json.loads((DETECTION / "thresholds.json").read_text())
detector_score_paths = [DETECTION / "detector_scores.csv"]
lower_args = []
if RUN_BINOCULARS:
    thresholds.update(json.loads((ZERO_SHOT / "zero_shot_thresholds.json").read_text()))
    detector_score_paths.append(ZERO_SHOT / "zero_shot_scores.csv")
    lower_args = ["--lower-is-ai", "binoculars_falcon7b"]
combined_thresholds = RUN / "detector_thresholds_combined.json"
combined_thresholds.write_text(json.dumps(thresholds, indent=2))
score_args = [item for path in detector_score_paths for item in ("--detector-scores", path)]
run("prepare_null_controls.py", "--cohort", COHORT, "--rewrites", REWRITES / "rewrites.jsonl",
    "--output", NULLS, "--include-rewrites", *score_args, "--thresholds", combined_thresholds, *lower_args)
display(pd.read_csv(NULLS / "null_control_comparison.csv").head(12))


## 11. Export e selezione delle annotazioni
Reference senza usare gli score e campione diagnostico arricchito separati per fonte; moduli per due annotatori, vuoti. Il target diagnostico di 100 si applica quando ci sono abbastanza coppie.


In [ ]:
score_tables = [SCORES / (stem + ".csv") for stem in [
    "lexical", "embedding", "bertscore", "nli", "aligned_cosine", "aligned_bertscore"]]
score_tables.append(STYLE / "guards.csv")
EXPORT = RUN / "continuous_suite"
score_args = [item for path in score_tables for item in ("--score", str(path))]
run("export_semantic_suite.py", "--pairs", target_pairs, *score_args, "--output", EXPORT)
continuous = pd.read_csv(EXPORT / "continuous_scores.csv")
meta = pd.DataFrame(read_jsonl(target_pairs))[["id", "domain", "generator"]]
continuous = continuous.merge(meta, on="id", validate="one_to_one")
continuous.to_csv(EXPORT / "continuous_scores_with_source_metadata.csv", index=False)
display(pd.read_csv(EXPORT / "humanizer_distributions.csv").head(20))
print("Riepilogo evasione:", DETECTION / "detection_summary.csv")
print("Risultati continui:", EXPORT / "continuous_scores_with_source_metadata.csv")
print("Stile e revisione:", STYLE)

run("prepare_semantic_audit_v2.py", "--input-pairs", target_pairs,
    "--scores", EXPORT / "continuous_scores_with_source_metadata.csv",
    "--output", RUN / "semantic_annotation_v2", "--reference-per-cell", 1,
    "--diagnostic-size", 100)
print(json.loads((RUN / "semantic_annotation_v2/sampling_manifest.json").read_text())["warning"])


## 12. Naturalezza sul nuovo pannello
Ordini AB e BA; preferenze discordanti restano discordanti. Massimo 768 tentativi POST per le tre comparazioni study, più 16 per la prova iniziale. Rate limit e copertura parziale sono registrati, senza inventare giudizi mancanti.


In [ ]:
if RUN_STUDY_JUDGE:
    count = 3 if MODE == "smoke" else 126
    # Frozen comparisons; at most 16 smoke / 256 study POST attempts per comparison.
    comparisons = [("qwen_humanize", "qwen14b_humanize"),
                   ("qwen_humanize", "dipper_context3"), ("dipper", "dipper_context3")]
    for a, b in comparisons:
        if not {a, b}.issubset(ACTIVE_SYSTEMS):
            continue
        destination = RUN / "naturalness" / (a + "__" + b)
        run("judge_naturalness.py", "--rewrites", REWRITES / "rewrites.jsonl",
            "--humanizer-a", a, "--humanizer-b", b, "--model", JUDGE_MODEL,
            "--max-pairs", count, "--max-requests", (16 if MODE == "smoke" else 256),
            "--execute", "--output", destination)
        manifest = json.loads((destination / "judge_run.json").read_text())
        print(a, b, manifest["status"], manifest["completed_valid_orders"], "/", manifest["expected_orders"])
        if manifest["status"] != "completed":
            print("Giudizio parziale:", manifest["blocked_reason"], "— i risultati mancanti rimangono mancanti.")


## 13. Archivio finale
Scaricare lo ZIP e restituirlo per l’analisi; verificare i manifest dei giudizi eventualmente parziali. Nessun peso, venv o secret è incluso.


In [ ]:
archive = shutil.make_archive(str(RUN), "zip", root_dir=RUN)
print("Archivio da scaricare:", archive)
print("Dimensione MiB:", round(Path(archive).stat().st_size / 2**20, 2))
from IPython.display import FileLink, display
display(FileLink(archive))
